# CLEA — provjeri GPU jednim pokretanjem
By Brzo i Lokalno

Pritisni **▶** uz ćeliju ispod i potvrdi pokretanje ako Colab zatraži. GPU se dodjeljuje prema dostupnosti.

Provjera ne preuzima modele, ne koristi privatne datoteke i ne pokreće trening. Pokazuje stvarni GPU i provjerava mali izračun na njemu. Dijeljenje ove bilježnice ne daje ChatGPT-u kontrolu nad sesijom.


In [ ]:
# CLEA GPU provjera — bez instalacije, preuzimanja modela ili treninga.
import json
import shutil
import subprocess
import time
from pathlib import Path


def check_clea_gpu():
    report = {
        "format": "clea7.gpu-preflight/1",
        "status": "not_checked",
        "training_started": False,
        "model_downloaded": False,
        "private_files_read": False,
        "network_requests": False,
        "cuda_computation_verified": False,
    }
    smi = shutil.which("nvidia-smi")
    if not smi:
        report.update(status="gpu_not_attached", message="GPU nije priključen. U Colabu odaberi Runtime → Change runtime type → GPU, zatim ponovo pokreni ovu ćeliju. Besplatan GPU možda trenutno nije dostupan.")
        return report
    try:
        probe = subprocess.run(
            [smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10, check=False,
        )
        if probe.returncode != 0:
            report.update(status="gpu_driver_unavailable", message="GPU drajver nije spreman. Provjeri da li je Colab povezan s GPU sesijom.")
            return report
        report["reported_hardware"] = probe.stdout.strip()[:1024]
        try:
            import torch
        except ImportError:
            report.update(status="cuda_library_missing", message="PyTorch nije dostupan u ovom okruženju. Ova provjera ne instalira ništa automatski.")
            return report
        if not torch.cuda.is_available():
            report.update(status="cuda_unavailable", message="GPU je prijavljen, ali CUDA izvršavanje nije dostupno.")
            return report
        device = torch.device("cuda:0")
        properties = torch.cuda.get_device_properties(device)
        report.update(
            gpu_name=properties.name,
            gpu_memory_bytes=int(properties.total_memory),
            gpu_memory_gib=round(properties.total_memory / (1024 ** 3), 2),
            torch_version=str(torch.__version__),
        )
        # A small actual GPU operation, not a model or performance benchmark.
        started = time.perf_counter()
        with torch.inference_mode():
            identity = torch.eye(64, device=device, dtype=torch.float32)
            values = torch.full((64, 64), 2.0, device=device)
            result = identity @ values
            torch.cuda.synchronize(device)
            verified = bool(torch.equal(result, values)) and float(result.sum().item()) == 8192.0
        del identity, values, result
        if not verified:
            report.update(status="cuda_check_failed", message="CUDA operacija nije dala očekivani rezultat.")
            return report
        report.update(
            status="gpu_verified",
            cuda_computation_verified=True,
            canary_elapsed_ms=round((time.perf_counter() - started) * 1000, 2),
            message="GPU je provjeren stvarnim izračunom. Ovo potvrđuje dostupnost GPU-a; nije trening niti dokaz kvaliteta CLEA modela.",
        )
    except subprocess.TimeoutExpired:
        report.update(status="gpu_probe_timed_out", message="GPU nije odgovorio u roku provjere. Pokušaj ponovo povezati Colab sesiju.")
    except Exception as error:
        report.update(status="gpu_check_error", error_type=type(error).__name__, message="Provjera GPU-a nije završena. Nije pokrenut trening niti instalacija.")
    return report


clea_gpu_report = check_clea_gpu()
report_path = Path("clea-gpu-report.json")
report_path.write_text(json.dumps(clea_gpu_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(clea_gpu_report["message"])
if clea_gpu_report.get("gpu_name"):
    print("GPU:", clea_gpu_report["gpu_name"])
    print("GPU memorija:", clea_gpu_report["gpu_memory_gib"], "GiB")
print("Izvještaj je spremljen u clea-gpu-report.json. Pristup ovoj sesiji nije podijeljen s ChatGPT-om.")
print(json.dumps(clea_gpu_report, ensure_ascii=False, indent=2))


Ako piše **GPU je provjeren**, pošalji u razgovor naziv GPU-a i memoriju iz rezultata. To su podaci potrebni za pripremu treninga; kvalitet modela još nije provjeren.

Ako GPU nije dodijeljen: **Runtime → Change runtime type → GPU** pa ponovo ▶. Ne trebaš plaćati ni unositi API ključeve.

[Službene informacije o dostupnosti GPU-a i dijeljenju bilježnice](https://research.google.com/colaboratory/faq.html)
